In [28]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
import re
import joblib
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [2]:
df_2024 = pd.read_csv('2024.csv')
df_2025 = pd.read_csv('2025.csv')

In [3]:
df = pd.concat([df_2024, df_2025], ignore_index=True)

In [4]:
columns_to_keep = [
    'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR', 'Attendance', 'Referee', 'HS', 'AS', 'HST', 'AST',
    'HHW', 'AHW', 'HC', 'AC', 'HF', 'AF', 'HFKC', 'AFKC', 'HO', 'AO', 'HY', 'AY',
    'HR', 'AR', 'IWH', 'IWD', 'IWA', 'WHH', 'WHD', 'WHA', 'B365H', 'B365D', 'B365A',
    'PH', 'PD', 'PA', 'AvgH', 'AvgD', 'AvgA', 'MaxH', 'MaxD', 'MaxA',
    'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'GB>2.5', 'GB<2.5', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5'
]

df = df[[col for col in columns_to_keep if col in df.columns]]

In [5]:
drop_cols = ['Time', 'Attendance', 'HHW', 'AHW', 'HO', 'AO', 'Div']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

def date_format_type(date_str):
    if not isinstance(date_str, str):
        return "not_a_string"
    patterns = {
        "%d/%m/%y": r"^\d{2}/\d{2}/\d{2}$",
        "%d/%m/%Y": r"^\d{2}/\d{2}/\d{4}$",
        "%Y-%m-%d": r"^\d{4}-\d{2}-\d{2}$",
        "%m-%d-%Y": r"^\d{2}-\d{2}-\d{4}$",
        "%Y/%m/%d": r"^\d{4}/\d{2}/\d{2}$",
    }
    for fmt, pat in patterns.items():
        if re.match(pat, date_str):
            return fmt
    return "unknown"

df['DateFormat'] = df['Date'].apply(date_format_type)
def parse_dates(row):
    date_str = row['Date']
    if isinstance(date_str, str):
        try:
            return pd.to_datetime(date_str, format='%d/%m/%y')
        except ValueError:
            try:
                return pd.to_datetime(date_str, format='%d/%m/%Y')
            except ValueError:
                return pd.NaT
    else:
        return pd.NaT

df['Date'] = df.apply(parse_dates, axis=1)
df = df.drop(columns=['DateFormat'], errors='ignore')
df = df[df['Date'] >= pd.Timestamp('2000-08-18')]
df = df.reset_index(drop=True)

def get_season(date):
    if pd.isnull(date):
        return np.nan
    year = date.year
    month = date.month
    if month >= 8: 
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"
df['Season'] = df['Date'].apply(get_season)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

odds_cols = [
    'IWH', 'IWD', 'IWA', 'WHH', 'WHD', 'WHA', 'B365H', 'B365D', 'B365A', 
    'B365>2.5', 'B365<2.5'
]
score_cols = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST',  
    'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]
for col in odds_cols + score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.drop_duplicates()

core_odds = ['IWH','IWD','IWA','WHH','WHD','WHA','B365H','B365D','B365A']
df = df.dropna(subset=[c for c in core_odds if c in df.columns])
df = df.reset_index(drop=True)

df['TotalGoals'] = df['FTHG'] + df['FTAG']
df['GoalsOver2_5'] = (df['TotalGoals'] > 2.5).astype(int)
df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
df['Home_2plus'] = (df['FTHG'] >= 2).astype(int)
df['Away_2plus'] = (df['FTAG'] >= 2).astype(int)

df = df.sort_values('Date')
df['HomeTeam_mean_FTHG'] = (
    df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).expanding().mean())
)
df['AwayTeam_mean_FTAG'] = (
    df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).expanding().mean())
)

df['HomeTeam_str'] = df['HomeTeam']
df['AwayTeam_str'] = df['AwayTeam']
df = pd.get_dummies(df, columns=['HomeTeam', 'AwayTeam'])
df = df.rename(columns={'HomeTeam_str': 'HomeTeam', 'AwayTeam_str': 'AwayTeam'})

def add_recent_form_features(df, n_matches=5):
    base = df.copy()
    base = base.sort_values('Date')
    home_df = base[['Date', 'HomeTeam', 'FTHG', 'FTAG']].rename(
        columns={'HomeTeam': 'Team', 'FTHG': 'GoalsFor', 'FTAG': 'GoalsAgainst'})
    away_df = base[['Date', 'AwayTeam', 'FTAG', 'FTHG']].rename(
        columns={'AwayTeam': 'Team', 'FTAG': 'GoalsFor', 'FTHG': 'GoalsAgainst'})
    results = pd.concat([home_df, away_df], ignore_index=True)
    results = results.sort_values(['Team', 'Date'])
   
    def get_points(row):
        return 3 if row['GoalsFor'] > row['GoalsAgainst'] else (1 if row['GoalsFor'] == row['GoalsAgainst'] else 0)
    results['Points'] = results.apply(get_points, axis=1)
    results['RollingGF'] = results.groupby('Team')['GoalsFor'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingGA'] = results.groupby('Team')['GoalsAgainst'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingPoints'] = results.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).sum())
  
    def get_form(row, team_col):
        team = row[team_col]
        date = row['Date']
        row_form = results[(results['Team'] == team) & (results['Date'] < date)].sort_values('Date').tail(1)
        if row_form.empty:
            return pd.Series([np.nan, np.nan, np.nan])
        return row_form[['RollingGF', 'RollingGA', 'RollingPoints']].values[0]
    base[['HomeRecentGF', 'HomeRecentGA', 'HomeRecentPts']] = base.apply(
        lambda row: get_form(row, 'HomeTeam'), axis=1, result_type='expand')
    base[['AwayRecentGF', 'AwayRecentGA', 'AwayRecentPts']] = base.apply(
        lambda row: get_form(row, 'AwayTeam'), axis=1, result_type='expand')
    return base

df = add_recent_form_features(df, n_matches=5)
df = df.dropna(subset=['HomeRecentGF', 'AwayRecentGF'])

df = df.sort_values('Date')
h2h_home_wins = []
team_stats = {}
home_positions = []
away_positions = []
for idx, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    match_date = row['Date']
    prev_matches = df[
        (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
         ((df['HomeTeam'] == away) & (df['AwayTeam'] == home)))
        & (df['Date'] < match_date)
    ].sort_values('Date', ascending=False).head(5)
    home_wins = ((prev_matches['HomeTeam'] == home) & (prev_matches['FTHG'] > prev_matches['FTAG'])).sum()
    h2h_home_wins.append(home_wins)

    league_table = []
    for team, stats in team_stats.items():
        league_table.append({
            'team': team,
            'points': stats['points'],
            'gd': stats['gd'],
            'scored': stats['scored']
        })
    table_df = pd.DataFrame(league_table)
    if not table_df.empty:
        table_df = table_df.sort_values(['points', 'gd', 'scored'], ascending=[False, False, False])
        table_df['position'] = range(1, len(table_df) + 1)
        home_pos = table_df[table_df['team'] == home]['position'].values[0] if home in table_df['team'].values else len(table_df) + 1
        away_pos = table_df[table_df['team'] == away]['position'].values[0] if away in table_df['team'].values else len(table_df) + 1
    else:
        home_pos = away_pos = 1
    home_positions.append(home_pos)
    away_positions.append(away_pos)
  
    home_goals = row['FTHG']
    away_goals = row['FTAG']
    for team in [home, away]:
        if team not in team_stats:
            team_stats[team] = {'points': 0, 'gd': 0, 'scored': 0}
    if home_goals > away_goals:
        team_stats[home]['points'] += 3
    elif home_goals < away_goals:
        team_stats[away]['points'] += 3
    else:
        team_stats[home]['points'] += 1
        team_stats[away]['points'] += 1
    team_stats[home]['gd'] += home_goals - away_goals
    team_stats[away]['gd'] += away_goals - home_goals
    team_stats[home]['scored'] += home_goals
    team_stats[away]['scored'] += away_goals

df['h2h_home_wins_last5'] = h2h_home_wins
df['home_league_position'] = home_positions
df['away_league_position'] = away_positions
df['position_diff'] = df['home_league_position'] - df['away_league_position']

N = 5
df['HomePts'] = np.where(df['FTHG'] > df['FTAG'], 3, np.where(df['FTHG'] == df['FTAG'], 1, 0))
df['AwayPts'] = np.where(df['FTAG'] > df['FTHG'], 3, np.where(df['FTAG'] == df['FTHG'], 1, 0))
df['HomeRecentPts'] = (
    df.groupby('HomeTeam')['HomePts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentPts'] = (
    df.groupby('AwayTeam')['AwayPts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['HomeGoalDiff'] = df['FTHG'] - df['FTAG']
df['AwayGoalDiff'] = df['FTAG'] - df['FTHG']
df['HomeRecentGoalDiff'] = (
    df.groupby('HomeTeam')['HomeGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentGoalDiff'] = (
    df.groupby('AwayTeam')['AwayGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentGoalDiff'] = df['HomeRecentGoalDiff'] - df['AwayRecentGoalDiff']
df['HomeRecentShotsOnTarget'] = (
    df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentShotsOnTarget'] = (
    df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentShotsOnTargetDiff'] = df['HomeRecentShotsOnTarget'] - df['AwayRecentShotsOnTarget']
df['PosDiff'] = df['home_league_position'] - df['away_league_position']

for col in ['B365>2.5', 'B365<2.5']:
    median = df[col].median()
    df[f'{col}_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(median)

df['B365>2.5_implied_prob'] = 1 / df['B365>2.5']
df['B365<2.5_implied_prob'] = 1 / df['B365<2.5']
df['OddsMargin'] = df['B365H'] / df['B365A']
df['OU_OddsMargin'] = df['B365>2.5'] / df['B365<2.5']
df['OverUnderRatio'] = df['B365>2.5'] / df['B365<2.5']

df['Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['EarlySeason'] = (df['Month'] <= 3).astype(int)

def rolling_ref_aggression(subdf):
    agg = subdf[['HY', 'AY', 'HR', 'AR']].shift(1).sum(axis=1)
    return agg.rolling(10, min_periods=1).mean()
df = df.sort_values('Date')
df['RefereeAggression'] = (
    df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)
)

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=['number']):
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'category']):
    df[col] = df[col].fillna(df[col].mode()[0])

df = df.drop_duplicates().reset_index(drop=True)

/var/folders/tk/jl3yfcz52s9020z4_8dbfkvw0000gn/T/ipykernel_27640/2276755120.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)


In [8]:
df.to_csv('combined_2024_2025.csv', index=False)

In [9]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,WHH,WHD,WHA,B365H,B365D,B365A,AvgH,AvgD,AvgA,MaxH,MaxD,MaxA,B365>2.5,B365<2.5,P>2.5,P<2.5,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Ipswich,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Newcastle,HomeTeam_Nott'm Forest,HomeTeam_Southampton,HomeTeam_Tottenham,HomeTeam_West Ham,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Ipswich,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Newcastle,AwayTeam_Nott'm Forest,AwayTeam_Southampton,AwayTeam_Tottenham,AwayTeam_West Ham,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression
0,2024-08-31,2,3,A,0,0,D,S Attwell,18,17,8,7,8,4,6,1,2,1,0,0,2.75,3.40,2.60,2.75,3.40,2.55,2.78,3.40,2.59,2.85,3.50,2.64,1.92,1.98,1.93,1.96,1.93,2.00,1.87,1.95,2024-25,2024,8,5,5,1,1,1,1,0.000000,1.000000,Everton,Bournemouth,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,0.0,3.0,6.0,1.0,1.0,5.0,0,1,1,0,0,3,-1,1,0.0,-1.0,0.0,22.0,18.0,3.0,0,0,0,0.520833,0.505051,1.078431,0.969697,0.969697,1,0,0.000000
1,2024-08-31,1,1,D,1,0,H,C Kavanagh,11,22,7,4,3,7,12,7,3,2,1,0,1.33,5.50,8.50,1.33,5.50,8.50,1.35,5.66,8.11,1.37,6.00,9.00,1.53,2.50,1.54,2.59,1.57,2.59,1.53,2.52,2024-25,2024,8,5,2,0,1,0,0,2.000000,3.000000,Arsenal,Brighton,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,2.0,0.0,6.0,3.0,0.0,5.0,0,3,3,0,1,1,0,0,0.0,-1.0,0.0,22.0,18.0,3.0,0,0,0,0.653595,0.400000,0.156471,0.612000,0.612000,1,0,0.000000
2,2024-08-31,3,1,H,1,0,H,J Smith,20,18,7,6,2,8,10,7,2,1,0,0,1.75,3.90,4.60,1.75,3.90,4.33,1.77,3.97,4.46,1.82,4.12,4.70,1.73,2.10,1.74,2.18,1.76,2.21,1.73,2.15,2024-25,2024,8,5,4,1,1,1,0,2.000000,0.000000,Brentford,Southampton,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,2.0,1.0,6.0,0.0,1.0,5.0,0,5,5,0,3,0,2,-2,0.0,-1.0,0.0,22.0,18.0,3.0,0,0,0,0.578035,0.476190,0.404157,0.823810,0.823810,1,0,0.000000
3,2024-08-31,1,1,D,1,1,D,L Smith,11,9,4,4,8,6,15,15,2,3,0,0,3.10,3.50,2.30,3.00,3.50,2.30,3.10,3.50,2.31,3.20,3.66,2.38,1.84,2.06,1.83,2.06,1.88,2.07,1.82,2.03,2024-25,2024,8,5,2,0,1,0,0,0.000000,0.000000,Ipswich,Fulham,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,0.0,2.0,6.0,0.0,1.0,5.0,0,7,7,0,1,1,0,0,0.0,-1.0,0.0,22.0,18.0,3.0,0,0,0,0.543478,0.485437,1.304348,0.893204,0.893204,1,0,0.000000
4,2024-08-31,1,1,D,1,1,D,S Hooper,16,11,5,

In [10]:
df_eval = df.sort_values('Date').tail(50).copy()

In [13]:
rf_home = joblib.load('../rf_home_2plus.pkl')
rf_away = joblib.load('../rf_away_2plus.pkl')

home_features = joblib.load('../home_2plus_features.pkl')
away_features = joblib.load('../away_2plus_features.pkl')

In [15]:
import numpy as np

def align_features(df, feature_list):
    for col in feature_list:
        if col not in df.columns:
            if "HomeTeam_" in col or "AwayTeam_" in col:
                df[col] = 0
            else:
                df[col] = np.nan
    return df[feature_list]

In [16]:
X_eval_home = align_features(df_eval, home_features)
X_eval_away = align_features(df_eval, away_features)

In [23]:
X_eval_home = X_eval_home.fillna(0)
X_eval_away = X_eval_away.fillna(0)

In [24]:
df_eval['pred_home_2plus'] = rf_home.predict(X_eval_home)
df_eval['proba_home_2plus'] = rf_home.predict_proba(X_eval_home)[:, 1]

df_eval['pred_away_2plus'] = rf_away.predict(X_eval_away)
df_eval['proba_away_2plus'] = rf_away.predict_proba(X_eval_away)[:, 1]

In [25]:
print(df_eval[['pred_home_2plus', 'proba_home_2plus', 'pred_away_2plus', 'proba_away_2plus']].head())

     pred_home_2plus  proba_home_2plus  pred_away_2plus  proba_away_2plus
219                0              0.44                0              0.38
220                0              0.46                1              0.54
221                0              0.25                1              0.52
222                1              0.58                0              0.41
223                0              0.35                0              0.41


In [32]:
print("HOME TEAM:")
print("Accuracy:", accuracy_score(df_eval['Home_2plus'], df_eval['pred_home_2plus']))
print("AUC:", roc_auc_score(df_eval['Home_2plus'], df_eval['proba_home_2plus']))
print("Confusion Matrix:\n", confusion_matrix(df_eval['Home_2plus'], df_eval['pred_home_2plus']))
print(classification_report(df_eval['Home_2plus'], df_eval['pred_home_2plus']))

print("\nAWAY TEAM:")
print("Accuracy:", accuracy_score(df_eval['Away_2plus'], df_eval['pred_away_2plus']))
print("AUC:", roc_auc_score(df_eval['Away_2plus'], df_eval['proba_away_2plus']))
print("Confusion Matrix:\n", confusion_matrix(df_eval['Away_2plus'], df_eval['pred_away_2plus']))
print(classification_report(df_eval['Away_2plus'], df_eval['pred_away_2plus']))

HOME TEAM:
Accuracy: 0.64
AUC: 0.680623973727422
Confusion Matrix:
 [[20  9]
 [ 9 12]]
              precision    recall  f1-score   support

           0       0.69      0.69      0.69        29
           1       0.57      0.57      0.57        21

    accuracy                           0.64        50
   macro avg       0.63      0.63      0.63        50
weighted avg       0.64      0.64      0.64        50


AWAY TEAM:
Accuracy: 0.6
AUC: 0.6316666666666667
Confusion Matrix:
 [[26  4]
 [16  4]]
              precision    recall  f1-score   support

           0       0.62      0.87      0.72        30
           1       0.50      0.20      0.29        20

    accuracy                           0.60        50
   macro avg       0.56      0.53      0.50        50
weighted avg       0.57      0.60      0.55        50

